# RAPTOR Chunking Arms — Experiment Runner (Colab)

Compares 5 leaf-chunking arms inside an otherwise **frozen** RAPTOR pipeline, isolating the
chunker as the only variable. Same gpt-oss model (via OpenRouter) is used for summarization
**and** QA across all arms, so results are directly comparable.

| Arm | Chunker |
|---|---|
| `token` | original RAPTOR sentence/token splitter (**baseline RAPTOR**) |
| `structure` | always-LLM structure chunking (char-offset map + per-section repair) |
| `ahc` | **Adaptive Hybrid Chunking** (structure score → route → repair) — *our architecture* |
| `semantic` | SBERT embedding-boundary chunking |
| `flat` | token chunks + FAISS, no tree (contextualizes tree value) |

Datasets & metrics (faithful to arXiv:2401.18059v1): **QASPER** → Answer-F1; **QuALITY** →
accuracy (+ HARD); **NarrativeQA** → ROUGE-L / BLEU-1/4 / METEOR. Retrieval = collapsed tree,
2000-token budget.

> **Tip:** use a GPU runtime (SBERT embeddings) and run the small **pilot** first to validate
> the whole pipeline before scaling up. Every LLM call is cached on disk, so reruns are cheap
> and the run is resumable.

## 1. Clone the repo and install the full stack

In [ ]:
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

## 2. Credentials + METEOR data
Get a free key at https://openrouter.ai/keys .

In [ ]:
import os, getpass
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

import nltk
for pkg in ('wordnet', 'omw-1.4', 'punkt'):
    nltk.download(pkg, quiet=True)
print('ready')

## 3. (Optional) Smoke-test the harness offline
All logic is unit-tested without network/model loads; run it to confirm the clone is intact.

In [ ]:
!python -m pytest tests/ -q

## 4. Configure the experiment
Start with the **pilot** (3 docs/dataset). To scale to the paper's subsets, bump
`subset_sizes` to `{'qasper': 50, 'quality': 50, 'narrativeqa': 25}` and add more `seeds`.
Swap `model` to the paid `openai/gpt-oss-120b` if the free tier rate-limits stall the run.

In [ ]:
from experiments.config import ExperimentConfig

cfg = ExperimentConfig(
    model='openai/gpt-oss-120b:free',     # -> 'openai/gpt-oss-120b' (paid) if rate-limited
    arms=['token', 'structure', 'ahc', 'semantic', 'flat'],
    datasets=['qasper', 'quality', 'narrativeqa'],
    subset_sizes={'qasper': 3, 'quality': 3, 'narrativeqa': 3},   # PILOT
    seeds=[0],
    retrieval_max_tokens=2000,   # paper's collapsed-tree main setting
    leaf_max_tokens=100,
    tau=0.5,                     # AHC routing threshold
    cache_dir='.llm_cache',
    results_dir='results',
)
cfg.to_dict()

## 5. Run
Builds one tree per (arm, dataset, doc) — SBERT embeddings + gpt-oss summaries — then answers
every question with the same gpt-oss reader and scores it. Progress prints per dataset/arm.
Interrupting is safe: cached calls make a rerun resume.

In [ ]:
from experiments import runner, report

results = runner.run(cfg, seed=cfg.seeds[0])
print('records:', len(results['records']))

## 6. Results: per-arm tables + AHC routing rate

In [ ]:
agg = report.aggregate(results['records'])
routing = report.routing_rate(results['records'])
print(report.to_markdown(agg, routing))

## 7. (Optional) Persist results + cache to Google Drive
So a disconnect doesn't lose the (expensive) LLM cache or results.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/raptor_runs && cp -r results .llm_cache /content/drive/MyDrive/raptor_runs/
# To resume later, copy .llm_cache back BEFORE running so cached LLM calls are reused.

## 8. Scale up to the full study
Re-run cells 4–6 with the larger config below (multiple seeds give mean±std + bootstrap CIs).
Run arms/datasets one at a time if the free tier is slow — results accumulate across runs via
the on-disk cache.

```python
cfg = ExperimentConfig(
    model='openai/gpt-oss-120b:free',
    subset_sizes={'qasper': 50, 'quality': 50, 'narrativeqa': 25},
    seeds=[0, 1, 2],
)
for s in cfg.seeds:
    runner.run(cfg, seed=s)
```